In [1]:
# -------------------------------
# 1️⃣ Install / Upgrade Libraries
# -------------------------------
!pip install -U transformers==5.3.0 datasets seqeval scikit-learn jsonlines

# -------------------------------
# 2️⃣ Imports
# -------------------------------
import json, jsonlines
import numpy as np
from sklearn.metrics import accuracy_score
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer

# -------------------------------
# 3️⃣ Label Definitions
# -------------------------------
labels = [
    "O",
    "B-TOPIC","I-TOPIC",
    "B-METHOD","I-METHOD",
    "B-CONCEPT","I-CONCEPT",
    "B-ALGORITHM","I-ALGORITHM",
    "B-OTHER","I-OTHER"
]
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

# -------------------------------
# 4️⃣ Load Dataset
# -------------------------------
data = []
with jsonlines.open("OS-cleaned_file.jsonl") as reader:
    for obj in reader:
        data.append(obj)

# -------------------------------
# 5️⃣ Convert Entities → BIO
# -------------------------------
def convert_to_tokens(sample):
    text = sample["input"]
    entities = json.loads(sample["output"])
    tokens = text.split()
    tags = ["O"] * len(tokens)

    for ent in entities:
        ent_text = ent["entity"].lower()
        label = ent["label"]
        ent_tokens = ent_text.split()
        for i in range(len(tokens)):
            window = tokens[i:i+len(ent_tokens)]
            window = [w.lower() for w in window]
            if window == ent_tokens:
                tags[i] = "B-" + label
                for j in range(1, len(ent_tokens)):
                    if i+j < len(tags):
                        tags[i+j] = "I-" + label
    return {"tokens": tokens, "ner_tags": tags}

processed = [convert_to_tokens(s) for s in data]
dataset = Dataset.from_list(processed)

# -------------------------------
# 6️⃣ Split Train/Test (90/10)
# -------------------------------
train_test = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test["train"]
test_dataset = train_test["test"]

# -------------------------------
# 7️⃣ Load RoBERTa
# -------------------------------
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# -------------------------------
# 8️⃣ Tokenization
# -------------------------------
def tokenize(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=256
    )
    word_ids = tokenized.word_ids()
    labels_ids = []
    for word_id in word_ids:
        if word_id is None:
            labels_ids.append(-100)
        else:
            tag = example["ner_tags"][word_id]
            labels_ids.append(label2id.get(tag, label2id["O"]))
    tokenized["labels"] = labels_ids
    return tokenized

train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

# -------------------------------
# 9️⃣ Training Arguments
# -------------------------------
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = TrainingArguments(
    output_dir="roberta_ner",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",       # <- use 'eval_strategy' instead of 'evaluation_strategy'
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50
)

# -------------------------------
# 1️⃣0️⃣ Metrics Function
# -------------------------------
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_labels, true_predictions = [], []
    for pred, lab in zip(predictions, labels):
        temp_pred, temp_lab = [], []
        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                temp_pred.append(id2label[p_i])
                temp_lab.append(id2label[l_i])
        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    # MICRO
    micro_precision = precision_score(true_labels, true_predictions)
    micro_recall = recall_score(true_labels, true_predictions)
    micro_f1 = f1_score(true_labels, true_predictions)

    # MACRO F1
    report = classification_report(true_labels, true_predictions, output_dict=True)
    macro_f1 = report["macro avg"]["f1-score"]

    # EXACT ACCURACY
    flat_true, flat_pred = [], []
    for t_seq, p_seq in zip(true_labels, true_predictions):
        flat_true.extend(t_seq)
        flat_pred.extend(p_seq)
    exact_accuracy = accuracy_score(flat_true, flat_pred)

    # PARTIAL ACCURACY (0.5 for partial match on entity type)
    partial_matches = 0
    total_tokens = 0
    for t_seq, p_seq in zip(true_labels, true_predictions):
        for t, p in zip(t_seq, p_seq):
            total_tokens += 1
            if t == p:
                partial_matches += 1
            elif t != "O" and p != "O" and t.split("-")[-1] == p.split("-")[-1]:
                partial_matches += 0.5
    partial_accuracy = partial_matches / total_tokens

    return {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "exact_accuracy": exact_accuracy,
        "partial_accuracy": partial_accuracy
    }

# -------------------------------
# 1️⃣1️⃣ Trainer
# -------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# -------------------------------
# 1️⃣2️⃣ Train Model
# -------------------------------
trainer.train()

# -------------------------------
# 1️⃣3️⃣ Evaluate Model
# -------------------------------
metrics = trainer.predict(test_dataset).metrics

print("\n===== Final Evaluation Metrics =====\n")
for k,v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.6 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=ad2b671d0ffab1d01f4be78981f363ef9e38394c4de5c0f3ff9be1f5d1add624
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



===== Final Evaluation Metrics =====

test_loss: 0.0792
test_micro_precision: 0.7603
test_micro_recall: 0.8140
test_micro_f1: 0.7863
test_macro_f1: 0.6434
test_exact_accuracy: 0.9779
test_partial_accuracy: 0.9779
test_runtime: 5.1875
test_samples_per_second: 60.3380
test_steps_per_second: 7.7110


In [ ]:
!rm -rf /content/roberta_ner